In [ ]:
import base64
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
token = secrets.get_secret("GITHUB_TOKEN")

username = "AnikMallick"
repo_name = "bn-proverbs-in-context"

repo_dir = Path("/kaggle/working") / repo_name

auth = base64.b64encode(
    f"{username}:{token}".encode()
).decode()

try:
    subprocess.run(
        [
            "git",
            "-c",
            f"http.https://github.com/.extraheader="
            f"AUTHORIZATION: basic {auth}",
            "clone",
            f"https://github.com/{username}/{repo_name}.git",
            str(repo_dir),
        ],
        check=True,
    )
finally:
    del token
    del auth

print("Repository cloned successfully.")

In [ ]:
%cd /kaggle/working/bn-proverbs-in-context

!pip install -e .

In [1]:
from pathlib import Path
from probad.utils import print_div
from probad.schema import TestItem, validate_items
import json
from typing import Any
import probad.builder as exp
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
DATA_PATH = Path("/kaggle/input/datasets/amallick0507/probad")

train_data = list(TestItem.load_items(DATA_PATH/"train.jsonl"))
test_data = list(TestItem.load_items(DATA_PATH/"test.jsonl"))
val_data = list(TestItem.load_items(DATA_PATH/"val.jsonl"))


source = DATA_PATH / "stopwords-bn.json"

try:
    data: Any = json.loads(source.read_text(encoding="utf-8-sig"))
except FileNotFoundError as exc:
    raise FileNotFoundError(f"Stopword file not found: {source}") from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Invalid JSON in stopword file {source}: "
        f"line {exc.lineno}, column {exc.colno}: {exc.msg}"
    ) from exc

BN_STOPWORDS =  frozenset([word.strip() for word in data])

print_div("train")
for issue in validate_items(train_data, BN_STOPWORDS):
    print(issue)

print_div("test")
for issue in validate_items(test_data, BN_STOPWORDS):
    print(issue)

print_div("val")
for issue in validate_items(val_data, BN_STOPWORDS):
    print(issue)

=============================================> train <==============================================
==============================================> test <==============================================
[WARNING] prv_088_1: en.correct_echoes_gloss: correct option reuses many literal-gloss words
[WARNING] prv_088_2: en.correct_echoes_gloss: correct option reuses many literal-gloss words
==============================================> val <===============================================


In [2]:
import os
import torch
import json
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

OUTPUT_DIR = "/kaggle/working/qwen3b-lora"

MAX_SEQ_LENGTH = 512

print(torch.cuda.get_device_name(0)), print(torch.cuda.device_count())  

Tesla T4
1


(None, None)

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map={"": 0},
#     torch_dtype=torch.float16,
#     trust_remote_code=True
# )

# model.config.use_cache = False

# model = prepare_model_for_kbit_training(
#     model,
#     use_gradient_checkpointing=True
# )

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16,
                                bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config,
                                             device_map={"": 0}, dtype=torch.float16)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [13]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

In [5]:
import json
from collections import Counter
import pandas as pd
from datasets import Dataset
import probad.builder as exp

TASKS = {"x1": exp.x1_correct_vs_literal, "x5": exp.x5_three_way}   # x2, x3, x4, x7 stay untrained (transfer tests)


def build(fn, item, lang, seed):
    """-> (prompt, gold letter in upper case) or None. Accepts the builder's 2-, 3- or 4-tuple."""
    out = fn(item, lang=lang, seed=seed)
    if out is None:
        return None
    out = tuple(out)
    if len(out) == 3 and not isinstance(out[2], (list, tuple)):   # (system, prompt, gold)
        out = out[1:]
    return out[0], str(out[1]).strip().upper()


def order_seeds(fn, item, lang, max_seed=300):
    """Seed 0 plus the first seeds that move the gold letter to every other position (same as scoring)."""
    first = build(fn, item, lang, 0)
    if first is None:
        return []
    n = sum(1 for L in "ABC" if f"\n{L}. " in "\n" + first[0])
    seeds, golds = [0], {first[1]}
    for s in range(1, max_seed):
        b = build(fn, item, lang, s)
        if b and b[1] not in golds:
            seeds.append(s); golds.add(b[1])
        if len(golds) >= n:
            break
    return seeds


def to_text(tok, prompt, system=None):
    """Exactly what the run notebook scores for a chat model: chat template, then the prompt's answer cue
    ("Answer:" / "উত্তর:") at the start of the reply. The next token is the letter."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return text + prompt.rstrip().rsplit("\n", 1)[-1]


def make_sft_dataset(items, tok, tasks=("x1", "x5"), langs=("en", "bn"), swap=True, out=None):
    """items: one split (dicts or TestItems). Returns a datasets.Dataset with prompt / completion (" A")
    for TRL's SFTTrainer, plus columns for checking (item_id, task, lang, order, gold)."""
    rows = []
    for it in items:
        it = it.model_dump() if hasattr(it, "model_dump") else it
        for task in tasks:
            for lang in langs:
                seeds = order_seeds(TASKS[task], it, lang) if swap else [0]
                for order, s in enumerate(seeds):
                    prompt, gold = build(TASKS[task], it, lang, s)
                    rows.append({"prompt": to_text(tok, prompt), "completion": " " + gold,
                                 "item_id": it["item_id"], "proverb_id": it["proverb"]["proverb_id"],
                                 "task": task, "lang": lang, "seed": s, "order": order, "gold": gold})
    df = pd.DataFrame(rows)

    n_tok = df["prompt"].map(lambda t: len(tok(t, add_special_tokens=False).input_ids))
    print(f"{len(df)} examples from {df.item_id.nunique()} items / {df.proverb_id.nunique()} proverbs | "
          f"tokens mean {n_tok.mean():.0f}, max {n_tok.max()}")
    print(df.groupby(["task", "lang"]).size().rename("examples").to_dict())
    print("gold letters per task (should be balanced):", {t: dict(Counter(g)) for t, g in df.groupby("task")["gold"]})
    if out:
        df.to_json(out, orient="records", lines=True, force_ascii=False)
        print("saved", out)
    return Dataset.from_pandas(df, preserve_index=False)

In [6]:
pid = lambda xs: {(x.model_dump() if hasattr(x, "model_dump") else x)["proverb"]["proverb_id"] for x in xs}
assert not (pid(train_data) | pid(val_data)) & pid(test_data), "train/val proverbs appear in test"

train_ds = make_sft_dataset(train_data, tokenizer, out="/kaggle/working/sft_train.jsonl")
val_ds = make_sft_dataset(val_data, tokenizer, out="/kaggle/working/sft_val.jsonl")

# SFTTrainer only needs these two columns
train_ds, val_ds = train_ds.select_columns(["prompt", "completion"]), val_ds.select_columns(["prompt", "completion"])

3000 examples from 300 items / 100 proverbs | tokens mean 263, max 511
{('x1', 'bn'): 600, ('x1', 'en'): 600, ('x5', 'bn'): 900, ('x5', 'en'): 900}
gold letters per task (should be balanced): {'x1': {'B': 600, 'A': 600}, 'x5': {'C': 600, 'B': 600, 'A': 600}}
saved /kaggle/working/sft_train.jsonl
750 examples from 75 items / 25 proverbs | tokens mean 267, max 513
{('x1', 'bn'): 150, ('x1', 'en'): 150, ('x5', 'bn'): 225, ('x5', 'en'): 225}
gold letters per task (should be balanced): {'x1': {'A': 150, 'B': 150}, 'x5': {'A': 150, 'B': 150, 'C': 150}}
saved /kaggle/working/sft_val.jsonl


In [7]:
train_ds[0], val_ds[0]

({'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nRajesh: Sumon trolled his ex-colleague online for weeks, and now she has posted all his old embarrassing messages.\nIpsita: Itti marle patkelti khete hoy.\n\nQuestion: What does Ipsita mean?\nA. Throw a brick at someone and a stone comes back at you.\nB. Sumon hurt her first, so he should have expected to be hurt back.\nAnswer:<|im_end|>\n<|im_start|>assistant\nAnswer:',
  'completion': ' B'},
 {'prompt': "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nAnkush: Montu went to rehab for six months and came back promising to change, and last night he was caught stealing from his own mother again.\nTuli: Koyla dhule moyla jay na.\n\nQuestion: What does Tuli mean?\nA. Montu's basic nature hasn't changed, whatever he promises.\nB. However much you wash a piece of coal, its blackness will never 

In [14]:
from trl import SFTConfig, SFTTrainer
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,          # effective batch 16 -> ~190 steps per epoch
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    fp16=True, bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},   # avoids the "no grad" problem with PEFT

    # data: prompt/completion columns, loss on the answer letter only
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    completion_only_loss=True,

    # keep the best checkpoint, not the last one
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,

    logging_steps=10,
    report_to="none",
    dataloader_num_workers=2,
    seed=0,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

In [15]:
b = trainer.data_collator([trainer.train_dataset[0]])
print(repr(tokenizer.decode(b["input_ids"][0][b["labels"][0] != -100])))



' B<|im_end|>'


In [16]:
for n, p in trainer.model.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()
print({str(p.dtype) for p in trainer.model.parameters() if p.requires_grad})

{'torch.float32'}


In [17]:
trainer.train()
trainer.save_model(OUTPUT_DIR + "/best")

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.015853,0.091546,0.006935,0.983957,214949.000000
100,0.000070,0.004442,0.002634,0.997326,427698.000000
150,0.000025,0.014591,0.003498,0.997326,635862.000000
200,0.000048,0.012651,0.002712,0.996658,847736.000000
250,0.000022,0.030349,0.001516,0.995989,1058374.000000
300,0.000073,0.015221,0.002649,0.997326,1271348.000000
350,0.000045,0.016954,0.001320,0.997995,1481507.000000
400,0.000026,0.018007,0.001146,0.997995,1686807.000000
450,0.000026,0.013356,0.001072,0.997326,1905691.000000
500,0.000024,0.013347,0.001081,0.997326,2118524.000000


In [18]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN_WRITE"))
api = HfApi()

REPO_ID = "anik-owl/qwen2.5-3b-instruct-bn-proverbs-lora"
ADAPTER_DIR = "/kaggle/working/qwen3b-lora/best"

api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(
    folder_path=ADAPTER_DIR, repo_id=REPO_ID, commit_message="QLoRA adapter (x1 + x5, train split)",
    ignore_patterns=["checkpoint-*", "optimizer.pt", "scheduler.pt", "rng_state*", "*.bin.tmp"],
)
print("https://huggingface.co/" + REPO_ID)

https://huggingface.co/anik-owl/qwen2.5-3b-instruct-bn-proverbs-lora


In [24]:
import json, platform, datetime
from pathlib import Path
import pandas as pd, torch, transformers, peft, trl

REPO_ID = "anik-owl/qwen2.5-3b-instruct-bn-proverbs-lora"

def data_mix(ds):
    """Counts per task / language / gold letter if those columns are still in the dataset."""
    cols = set(ds.column_names)
    if not {"task", "lang"} <= cols:
        return {"examples": len(ds)}
    df = ds.to_pandas()
    return {"examples": len(df), "items": int(df["item_id"].nunique()) if "item_id" in cols else None,
            "proverbs": int(df["proverb_id"].nunique()) if "proverb_id" in cols else None,
            "per_task_lang": {f"{t}_{l}": int(n) for (t, l), n in df.groupby(["task", "lang"]).size().items()},
            "gold_letters": df["gold"].value_counts().to_dict() if "gold" in cols else None}

log = pd.DataFrame(trainer.state.log_history)
log.to_csv(OUT / "train_log.csv", index=False)
final = log.dropna(subset=["train_runtime"]).iloc[-1].to_dict() if "train_runtime" in log else {}
ev = log.dropna(subset=["eval_loss"]) if "eval_loss" in log else pd.DataFrame()

model_cfg = trainer.model.config
qcfg = getattr(model_cfg, "quantization_config", None)
info = {
    "saved_at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "base_model": "Qwen/Qwen2.5-3B-Instruct",
    "quantization": qcfg.to_dict() if hasattr(qcfg, "to_dict") else qcfg,
    "lora": {k: (sorted(v) if isinstance(v, set) else v)
             for k, v in trainer.model.peft_config["default"].to_dict().items()},
    "trainable_params": sum(p.numel() for p in trainer.model.parameters() if p.requires_grad),
    "total_params": sum(p.numel() for p in trainer.model.parameters()),
    "training_args": {k: v for k, v in trainer.args.to_dict().items()
                      if not k.startswith(("hub_", "push_to_hub", "logging_dir"))},
    "data": {
        "description": "x1 (correct vs literal) + x5 (3 options), en + bn conversations, all option orders "
                       "(gold in every position); prompt = scoring text ending in the answer cue, "
                       "completion = answer letter; loss on the completion only",
        "train": data_mix(train_ds_full if "train_ds_full" in globals() else trainer.train_dataset),
        "val": data_mix(val_ds_full if "val_ds_full" in globals() else trainer.eval_dataset),
        "test_proverbs_overlap": 0,        # checked with the assert before training
    },
    "result": {
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_eval_loss": trainer.state.best_metric,
        "first_eval_loss": float(ev["eval_loss"].iloc[0]) if len(ev) else None,
        "last_eval_loss": float(ev["eval_loss"].iloc[-1]) if len(ev) else None,
        "global_steps": trainer.state.global_step, "epochs_run": trainer.state.epoch,
        "train_runtime_min": round(final.get("train_runtime", 0) / 60, 1),
        "final_train_loss": final.get("train_loss"),
    },
    "environment": {
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "peft": peft.__version__, "trl": trl.__version__,
        "bitsandbytes": __import__("bitsandbytes").__version__,
    },
}
json.dump(info, open(OUT / "train_info.json", "w"), indent=2, default=str)
print(json.dumps({k: info[k] for k in ("base_model", "result", "data")}, indent=2, default=str))

{
  "base_model": "Qwen/Qwen2.5-3B-Instruct",
  "result": {
    "best_checkpoint": "/kaggle/working/qwen3b-lora/checkpoint-100",
    "best_eval_loss": 0.00444174325093627,
    "first_eval_loss": 0.09154625982046127,
    "last_eval_loss": 0.013427305035293102,
    "global_steps": 564,
    "epochs_run": 3.0,
    "train_runtime_min": 118.5,
    "final_train_loss": 0.22937476248758776
  },
  "data": {
    "description": "x1 (correct vs literal) + x5 (3 options), en + bn conversations, all option orders (gold in every position); prompt = scoring text ending in the answer cue, completion = answer letter; loss on the completion only",
    "train": {
      "examples": 3000
    },
    "val": {
      "examples": 747
    },
    "test_proverbs_overlap": 0
  }
}


In [25]:
from huggingface_hub import HfApi
api = HfApi()
for f in ["train_info.json", "train_log.csv", "train_curve.png"]:
    if (OUT / f).exists():
        api.upload_file(path_or_fileobj=str(OUT / f), path_in_repo=f, repo_id=REPO_ID)
        print("uploaded", f)

uploaded train_info.json
uploaded train_log.csv
uploaded train_curve.png
